In [7]:
import subprocess
subprocess.run(["pip", "install", "psycopg2-binary", "sqlalchemy"], check=True)
print("Dependencies installed!")

Dependencies installed!


# 03 — Products Transformation
Reads raw.products from PostgreSQL, validates and cleans the data using Pandas,
then loads clean records into staging.products.

In [8]:
import os
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime, timezone

engine = create_engine(
    f"postgresql://{os.environ['POSTGRES_USER']}:{os.environ['POSTGRES_PASSWORD']}"
    f"@{os.environ['POSTGRES_HOST']}:{os.environ['POSTGRES_PORT']}/{os.environ['POSTGRES_DB']}"
)

print("Connected!")

Connected!


## Step 1 — Load raw products

In [9]:
df = pd.read_sql("SELECT * FROM raw.products;", engine)

print(f"Loaded {len(df):,} raw product records")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData types:\n{df.dtypes}")
df.head()

Loaded 88 raw product records

Columns: ['product_id', 'product_name', 'category', 'price', 'batch_id', 'created_at']

Data types:
product_id              object
product_name            object
category                object
price                  float64
batch_id                 int64
created_at      datetime64[ns]
dtype: object


,product_id,product_name,category,price,batch_id,created_at
0,PROD9311,Yoga Mat,Sports & Fitness,14110.55,4,2026-04-18 03:37:30.534
1,PROD8545,Resistance Bands,Sports & Fitness,1098.59,4,2026-04-18 03:37:30.534
2,PROD7597,Cookbook,Books,10672.45,4,2026-04-18 03:37:30.534
3,PROD2632,Bed Sheets,Home & Kitchen,11770.66,4,2026-04-18 03:37:30.534
4,PROD6019,Smart Watch,Electronics,11784.85,4,2026-04-18 03:37:30.534


## Step 2 — Validate

In [10]:
required_fields = ["product_id", "product_name", "category", "price"]

print(" NULL CHECKS ")
for field in required_fields:
    null_count = df[field].isnull().sum()
    status = "PASS" if null_count == 0 else "FAIL"
    print(f"  [{status}] {field}: {null_count} nulls")

# Price must be greater than zero
invalid_price = (df["price"] <= 0).sum()
print(f"\n PRICE CHECK ")
print(f"  [{'PASS' if invalid_price == 0 else 'FAIL'}] prices <= 0: {invalid_price}")

# Duplicate check
duplicates = df["product_id"].duplicated().sum()
print(f"\n DUPLICATE CHECK ")
print(f"  [{'PASS' if duplicates == 0 else 'FAIL'}] product_id duplicates: {duplicates}")

# Category values
print(f"\n CATEGORY VALUES ")
print(df["category"].value_counts().to_string())

 NULL CHECKS 
  [PASS] product_id: 0 nulls
  [PASS] product_name: 0 nulls
  [PASS] category: 0 nulls
  [PASS] price: 0 nulls

 PRICE CHECK 
  [PASS] prices <= 0: 0

 DUPLICATE CHECK 
  [PASS] product_id duplicates: 0

 CATEGORY VALUES 
category
Books               21
Sports & Fitness    20
Home & Kitchen      17
Electronics         15
Clothing            15


## Step 3 — Clean and Transform

In [11]:
df_clean = df.copy()

# Normalize product_name to title case
df_clean["product_name"] = df_clean["product_name"].str.strip().str.title()

# Normalize category to title case
df_clean["category"] = df_clean["category"].str.strip().str.title()

# Round price to 2 decimal places
df_clean["price"] = df_clean["price"].round(2)

# Add loaded_at timestamp
df_clean["loaded_at"] = datetime.now(timezone.utc)

print(f"Cleaned {len(df_clean):,} product records")
df_clean.head()

Cleaned 88 product records


,product_id,product_name,category,price,batch_id,created_at,loaded_at
0,PROD9311,Yoga Mat,Sports & Fitness,14110.55,4,2026-04-18 03:37:30.534,2026-04-22 17:17:39.625874+00:00
1,PROD8545,Resistance Bands,Sports & Fitness,1098.59,4,2026-04-18 03:37:30.534,2026-04-22 17:17:39.625874+00:00
2,PROD7597,Cookbook,Books,10672.45,4,2026-04-18 03:37:30.534,2026-04-22 17:17:39.625874+00:00
3,PROD2632,Bed Sheets,Home & Kitchen,11770.66,4,2026-04-18 03:37:30.534,2026-04-22 17:17:39.625874+00:00
4,PROD6019,Smart Watch,Electronics,11784.85,4,2026-04-18 03:37:30.534,2026-04-22 17:17:39.625874+00:00


## Step 4 — Create staging table and load

In [12]:
with engine.connect() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS staging.products (
            product_id      VARCHAR PRIMARY KEY,
            product_name    VARCHAR NOT NULL,
            category        VARCHAR NOT NULL,
            price           NUMERIC(10, 2),
            batch_id        INTEGER,
            created_at      TIMESTAMP WITH TIME ZONE,
            loaded_at       TIMESTAMP WITH TIME ZONE
        );
    """))
    conn.commit()

print("staging.products table ready!")

staging.products table ready!


In [13]:
df_clean.to_sql(
    name="products",
    schema="staging",
    con=engine,
    if_exists="append",
    index=False,
)

print(f"Loaded {len(df_clean):,} products into staging.products")

Loaded 88 products into staging.products


## Step 5 — Verify

In [14]:
df_staging = pd.read_sql("SELECT * FROM staging.products;", engine)

print(f"staging.products row count: {len(df_staging):,}")
print(f"\nNull counts:\n{df_staging.isnull().sum()}")
df_staging.head()

staging.products row count: 88

Null counts:
product_id      0
product_name    0
category        0
price           0
batch_id        0
created_at      0
loaded_at       0
dtype: int64


,product_id,product_name,category,price,batch_id,created_at,loaded_at
0,PROD9311,Yoga Mat,Sports & Fitness,14110.55,4,2026-04-18 03:37:30.534000+00:00,2026-04-22 17:17:39.625874+00:00
1,PROD8545,Resistance Bands,Sports & Fitness,1098.59,4,2026-04-18 03:37:30.534000+00:00,2026-04-22 17:17:39.625874+00:00
2,PROD7597,Cookbook,Books,10672.45,4,2026-04-18 03:37:30.534000+00:00,2026-04-22 17:17:39.625874+00:00
3,PROD2632,Bed Sheets,Home & Kitchen,11770.66,4,2026-04-18 03:37:30.534000+00:00,2026-04-22 17:17:39.625874+00:00
4,PROD6019,Smart Watch,Electronics,11784.85,4,2026-04-18 03:37:30.534000+00:00,2026-04-22 17:17:39.625874+00:00
